Load packages

In [2]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely import Polygon
import math

from compactness_metric_functions import *

import maup

import networkx as nx

from gerrychain import (GeographicPartition, Partition, Graph, MarkovChain,
                        proposals, updaters, constraints, accept, Election)
from gerrychain.proposals import recom, propose_random_flip
from functools import partial
import pandas
from gerrychain.tree import recursive_tree_part

from gerrychain.metrics import efficiency_gap, mean_median, polsby_popper, partisan_bias

import os

from gerrychain.constraints.contiguity import contiguous_components, contiguous

from gerrychain.updaters import cut_edges

from gerrychain.tree import bipartition_tree, find_balanced_edge_cuts_memoization

from gerrychain.updaters import num_spanning_trees

import numpy as np

import pandas as pd

import random

Load files

In [ ]:
CON = gpd.read_file("./NC_Processed/nc_cong_adopted_2025/NCGA_CCM-2 - Shapefile/CCM-2.shp")
Precincts = gpd.read_file("./NC_Processed/output/NC_Processed_Precincts_eveomett.shp")

graph = Graph.from_json("./NC_Processed/output/NC_Processed_Precincts_eveomett.json")


# Compactness
Setup

In [62]:
Precincts.keys()

Index(['PREC_ID20', 'COUNTY_ID2', 'COUNTY_NAM', 'ENR_DESC20', 'CD', 'SEND',
       'HDIST', 'TOTPOP', 'NH_BLACK', 'NH_OTHER', 'NH_WHITE', 'HISP', 'VAP',
       'HVAP', 'WVAP', 'BVAP', 'OTHERVAP', 'AGR16D', 'AGR16R', 'AGR20D',
       'AGR20R', 'ATG16D', 'ATG16R', 'ATG20D', 'ATG20R', 'AUD16D', 'AUD16R',
       'AUD20D', 'AUD20R', 'GOV16D', 'GOV16O', 'GOV16R', 'GOV20D', 'GOV20O',
       'GOV20R', 'INS16D', 'INS16R', 'INS20D', 'INS20R', 'LAB16D', 'LAB16O',
       'LAB16R', 'LAB20D', 'LAB20R', 'LTG16D', 'LTG16O', 'LTG16R', 'LTG20D',
       'LTG20R', 'PRE16D', 'PRE16O', 'PRE16R', 'PRE20D', 'PRE20O', 'PRE20R',
       'SAC16D', 'SAC16O', 'SAC16R', 'SAC18D', 'SAC18O', 'SAC18R', 'SAC20D',
       'SAC20R', 'SOS16D', 'SOS16R', 'SOS20D', 'SOS20R', 'SPI16D', 'SPI16R',
       'SPI20D', 'SPI20R', 'SSC16D', 'SSC16R', 'SSC18D', 'SSC18R', 'SSC20D',
       'SSC20R', 'TRE16D', 'TRE16R', 'TRE20D', 'TRE20R', 'USS16D', 'USS16O',
       'USS16R', 'USS20D', 'USS20O', 'USS20R', 'geometry'],
      dtype='object')

In [4]:
def polsby_popper_gdf(gdf):
    return 4*math.pi * gdf.area/(gdf.length**2)

def count_spanning(graph):
    laplacian = nx.laplacian_matrix(graph)
    L = np.delete(np.delete(laplacian.todense(), 0, 0), 1, 1)
    return np.linalg.slogdet(L)[1]

def county_splits(partition, df=Precincts):
    df["current"] = df.index.map(partition.assignment)

    counties = sum(df.groupby("COUNTY_ID2")['current'].nunique()>1)
    return counties

election_names = [
    "PRE",
    "USS",
    "GOV"
]

num_elections = len(election_names)

election_columns = [
  ['PRE20R','PRE20D'],
  ['USS20R','USS20D'],
  ['GOV20R','GOV20D']
]

my_updaters = {
    "population": updaters.Tally("TOTPOP", alias="population"),
    "cut_edges": cut_edges,
    "PP":polsby_popper,
    "county_splits": county_splits
}

elections = [
    Election(
        election_names[i],
        {"Democratic": election_columns[i][1], "Republican": election_columns[i][0]},
    )
    for i in range(num_elections)
]

election_updaters = {election.name: election for election in elections}
for node in graph.nodes():
    graph.nodes()[node]["NON_NH_BLACK"] = graph.nodes()[node]["TOTPOP"] - graph.nodes()[node]["NH_BLACK"]

my_updaters.update({"NH_BLACK": "NH_BLACK", "NON_NH_BLACK": "NON_NH_BLACK"})

# save percentages

my_updaters.update(election_updaters)


## District-level compactness scores

In [64]:
CON_from_Precincts = Precincts.dissolve(by='CD')
SLDU_from_Precincts = Precincts.dissolve(by='SEND')
SLDL_from_Precincts = Precincts.dissolve(by='HDIST')

In [65]:
plans = [CON_from_Precincts,SLDU_from_Precincts,SLDL_from_Precincts]

for plan in plans: 
    plan['% Black'] = plan['NH_BLACK']/plan['TOTPOP']
    plan['PP'] = polsby_popper_gdf(plan)
    plan['CH'] = c_hull_ratio(plan)
    plan['R'] = 0
    plan['LW'] = 0
    plan['P'] = plan.length
    for ind, row in plan.iterrows():
        plan.loc[ind,'R']=(row['geometry'].area/(math.pi * make_circle(list(row['geometry'].convex_hull.exterior.coords))[2]**2))
        
        
        outside = list(row['geometry'].convex_hull.envelope.exterior.coords)

        o_len = max([x[0] for x in outside]) - min([x[0] for x in outside])

        o_wid = max([x[1] for x in outside]) - min([x[1] for x in outside])

        lw = min(o_len/o_wid,o_wid/o_len)
        
        plan.loc[ind,'LW'] = lw

C:\Users\angel\AppData\Local\Temp\ipykernel_6544\2964938161.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.4915659937779693' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  plan.loc[ind,'R']=(row['geometry'].area/(math.pi * make_circle(list(row['geometry'].convex_hull.exterior.coords))[2]**2))
C:\Users\angel\AppData\Local\Temp\ipykernel_6544\2964938161.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.7750171891091184' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  plan.loc[ind,'LW'] = lw
C:\Users\angel\AppData\Local\Temp\ipykernel_6544\2964938161.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.2626613733706847' has dtype incompatible with 

In [66]:
CON_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./NC_Stats/NC_Compactness_CON_eveomett.csv")
SLDU_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./NC_Stats/NC_Compactness_SLDU_eveomett.csv")
SLDL_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./NC_Stats/NC_Compactness_SLDL_eveomett.csv")

In [67]:
graph.nodes()[0]

{'boundary_node': False,
 'area': 131376438.59656279,
 'PREC_ID20': '01',
 'COUNTY_ID2': 1,
 'COUNTY_NAM': 'ALAMANCE',
 'ENR_DESC20': 'PATTERSON',
 'CD': '9',
 'SEND': '25',
 'HDIST': '64',
 'TOTPOP': 5135.0,
 'NH_BLACK': 309.0,
 'NH_OTHER': 155.0,
 'NH_WHITE': 4420.0,
 'HISP': 251.0,
 'VAP': 4066.0,
 'HVAP': 167.0,
 'WVAP': 3544.0,
 'BVAP': 237.0,
 'OTHERVAP': 118.0,
 'AGR16D': 379.0,
 'AGR16R': 1935.0,
 'AGR20D': 485,
 'AGR20R': 2401,
 'ATG16D': 519.0,
 'ATG16R': 1784.0,
 'ATG20D': 618,
 'ATG20R': 2266,
 'AUD16D': 534.0,
 'AUD16R': 1748.0,
 'AUD20D': 608,
 'AUD20R': 2247,
 'GOV16D': 499.0,
 'GOV16O': 45.0,
 'GOV16R': 1806.0,
 'GOV20D': 661,
 'GOV20O': 32,
 'GOV20R': 2209,
 'INS16D': 487.0,
 'INS16R': 1795.0,
 'INS20D': 561,
 'INS20R': 2304,
 'LAB16D': 392.0,
 'LAB16O': 1.0,
 'LAB16R': 1901.0,
 'LAB20D': 573,
 'LAB20R': 2293,
 'LTG16D': 422.0,
 'LTG16O': 60.0,
 'LTG16R': 1839.0,
 'LTG20D': 556,
 'LTG20R': 2336,
 'PRE16D': 411.0,
 'PRE16O': 75.0,
 'PRE16R': 1865.0,
 'PRE20D': 566,
 'PR

# Cut Edges

In [5]:
CONPart = GeographicPartition(graph,"CD",my_updaters)
SLDUPart = GeographicPartition(graph,"SEND",my_updaters)
SLDLPart = GeographicPartition(graph,"HDIST",my_updaters)

In [69]:
"""
subgraph12 = graph.subgraph(CONPart.parts["12"])
subgraph13 = graph.subgraph(CONPart.parts["13"])

S12 = [c for c in nx.connected_components(subgraph12)]
S13 = [c for c in nx.connected_components(subgraph13)]

graph.nodes()[1126]["CD"] = '8'
graph.nodes()[1668]["CD"] = '13'
"""

'\nsubgraph12 = graph.subgraph(CONPart.parts["12"])\nsubgraph13 = graph.subgraph(CONPart.parts["13"])\n\nS12 = [c for c in nx.connected_components(subgraph12)]\nS13 = [c for c in nx.connected_components(subgraph13)]\n\ngraph.nodes()[1126]["CD"] = \'8\'\ngraph.nodes()[1668]["CD"] = \'13\'\n'

In [10]:
len(SLDLPart.parts.keys())

120

In [11]:
for i in range(1,121):
    print(i,nx.is_connected(graph.subgraph(SLDLPart.parts[str(i)])))

1 True
2 True
3 True
4 True
5 True
6 True
7 True
8 True
9 True
10 True
11 True
12 True
13 True
14 True
15 True
16 True
17 True
18 True
19 True
20 True
21 True
22 True
23 True
24 True
25 True
26 True
27 True
28 True
29 True
30 True
31 True
32 True
33 True
34 True
35 True
36 True
37 True
38 True
39 True
40 True
41 True
42 True
43 True
44 True
45 True
46 True
47 True
48 True
49 True
50 True
51 True
52 True
53 True
54 True
55 True
56 True
57 True
58 True
59 True
60 True
61 True
62 True
63 True
64 True
65 True
66 True
67 True
68 True
69 True
70 True
71 True
72 True
73 True
74 True
75 True
76 True
77 True
78 True
79 True
80 True
81 True
82 True
83 True
84 True
85 True
86 True
87 True
88 True
89 True
90 True
91 True
92 True
93 True
94 True
95 True
96 True
97 True
98 True
99 True
100 True
101 True
102 True
103 True
104 True
105 True
106 True
107 True
108 True
109 True
110 True
111 True
112 True
113 True
114 True
115 True
116 True
117 True
118 True
119 True
120 True


In [71]:
"""
#for i in range(1,51):
    #print(i,nx.is_connected(graph.subgraph(SLDUPart.parts[str(i)])))

subgraph14 = graph.subgraph(SLDUPart.parts["14"])
subgraph35 = graph.subgraph(SLDUPart.parts["35"])

S14 = [c for c in nx.connected_components(subgraph14)]
S35 = [c for c in nx.connected_components(subgraph35)]

print([len(x) for x in S14])
print([len(x) for x in S35])

#print(S35[1])
print(S14[1])

graph.nodes()[614]["SEND"] = '34'

graph.nodes()[1799]["SEND"] = '14'

graph.nodes()[1677]

for node in graph.nodes():
    if graph.nodes()[node]["PREC_ID20"] == '17-02':
        print(node)

graph.nodes()[1799]
"""

'\n#for i in range(1,51):\n    #print(i,nx.is_connected(graph.subgraph(SLDUPart.parts[str(i)])))\n\nsubgraph14 = graph.subgraph(SLDUPart.parts["14"])\nsubgraph35 = graph.subgraph(SLDUPart.parts["35"])\n\nS14 = [c for c in nx.connected_components(subgraph14)]\nS35 = [c for c in nx.connected_components(subgraph35)]\n\nprint([len(x) for x in S14])\nprint([len(x) for x in S35])\n\n#print(S35[1])\nprint(S14[1])\n\ngraph.nodes()[614]["SEND"] = \'34\'\n\ngraph.nodes()[1799]["SEND"] = \'14\'\n\ngraph.nodes()[1677]\n\nfor node in graph.nodes():\n    if graph.nodes()[node]["PREC_ID20"] == \'17-02\':\n        print(node)\n\ngraph.nodes()[1799]\n'

In [72]:
CONPart = GeographicPartition(graph,"CD",my_updaters)
SLDUPart = GeographicPartition(graph,"SEND",my_updaters)

In [73]:
print(contiguous(CONPart))
print(contiguous(SLDUPart))
print(contiguous(SLDLPart))

False
False
True


In [74]:
ideal_con_population = sum(CONPart["population"].values()) / len(CONPart)
ideal_sldu_population = sum(SLDUPart["population"].values()) / len(SLDUPart)
ideal_sldl_population = sum(SLDLPart["population"].values()) / len(SLDLPart)

proposed_plans = [CONPart, SLDUPart, SLDLPart]
plan_names = ["CON","SLDU","SLDL"]

clist = ['green','hotpink','orange']

In [75]:
summary = [[] for plan in proposed_plans]


for i in range(len(proposed_plans)):
    #print("Dem Seats:", plan_names[i],proposed_plans[i]['PRS'].wins("Democratic"))
    print(plan_names[i])
    
    print("Cut Edges:", plan_names[i],len(proposed_plans[i]['cut_edges']))

    summary[i].append(len(proposed_plans[i]['cut_edges']))
    
    temp = 0
    for dist in proposed_plans[i].parts:
        tgraph = proposed_plans[i].subgraphs[dist]
        temp += count_spanning(tgraph)


    print("Spanning trees:",plan_names[i],temp)
    summary[i].append(temp)
    print("Mean Polsby_Popper:", plan_names[i],np.mean(list(polsby_popper(proposed_plans[i]).values())))
    summary[i].append(np.mean(list(polsby_popper(proposed_plans[i]).values())))
    print("County Splits:", plan_names[i],county_splits(proposed_plans[i]))
    summary[i].append(county_splits(proposed_plans[i]))

    
    print("Mean Population Deviation",plan_names[i],np.mean([abs((x-ideal_con_population))/ideal_con_population for x in list(proposed_plans[i]['population'].values())]))
    summary[i].append(np.mean([abs((x-ideal_con_population))/ideal_con_population for x in list(proposed_plans[i]['population'].values())]))

CON
Cut Edges: CON 735
Spanning trees: CON -inf
Mean Polsby_Popper: CON 0.2658147889501084
County Splits: CON 11
Mean Population Deviation CON 0.0031559047604815616
SLDU
Cut Edges: SLDU 1409
Spanning trees: SLDU -inf
Mean Polsby_Popper: SLDU 0.31813953267413814
County Splits: SLDU 15
Mean Population Deviation SLDU 0.7200000000000002
SLDL
Cut Edges: SLDL 2058
Spanning trees: SLDL 2647.5798794798397
Mean Polsby_Popper: SLDL 0.3512522356343281
County Splits: SLDL 36
Mean Population Deviation SLDL 0.8833333333333333


Figure out why spanning trees are -inf for the first two?

In [76]:
summary_df = pd.DataFrame(summary, columns=['Cut Edges', 'Spanning Trees', 'Mean PP', 'County Splits', 'Mean Pop Deviation'], index=plan_names)


In [77]:
summary_df.to_csv("./NC_Stats/NC_Compactness_Summary_eveomett.csv")

# Partisan Symmetry
Utilizes 2020 Data

In [78]:
#Helper functions
def plot_symmetry_with_mean_overhaul_rb(pvec,mean,xl=0,xu=1,yl=0,yu=1):
    
    n=5000
    l=len(pvec)
    pvec = np.array(sorted(pvec))
    seats = []
    votes = []
    small = pvec[0]
    large = pvec[-1]
    
    gap = large - small
    
    
    
    #mean = np.mean(pvec)
    
    lvec = pvec - mean*np.ones([1,l])
    
    
    
    for t in range(n):
        tvec = lvec + (t/n)*np.ones([1,l])
        votes.append(t/n)#votes.append(np.mean(tvec))
        seats.append(sum(sum(tvec>=.5))/l)
        
    
 
    #print(lvec,tvec,seats[400])
    
    plt.figure(figsize=(8,8) )   
    plt.plot(votes,seats,linewidth = 3,color='r')
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')

    #plt.plot([.5],[.5],'ro', markersize=10)
    plt.plot([mean],[sum(pvec>=.5)/l],'r*', markersize=20)

    plt.xlabel("Dem Vote %")
    plt.ylabel("Dem Seat %")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)
    
    plt.xlim([xl,xu])
    plt.ylim([yl,yu])

    plt.title("Seats -- Votes")
    
    

    plt.show()
    
    fvotes = [1-x for x in votes]
    fseats = [1-x for x in seats]
    
    plt.figure(figsize=(8,8) )   
    plt.plot(votes,seats,linewidth = 1,color='blue')
    plt.plot(fvotes,fseats,linewidth = 1,color='red')
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')
    plt.fill_between(votes,seats,list(reversed(fseats)),color='gray')

    #plt.plot([.5],[.5],'ro', markersize=10)
    #plt.plot([mean],[sum(pvec>=.5)/l],'g*', markersize=10)

    plt.xlabel("Dem Vote %")
    plt.ylabel("Dem Seat %")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)
    
    plt.xlim([xl,xu])
    plt.ylim([yl,yu])

    plt.title("Seats -- Votes: Symmetry Gaps")    
  

def plot_lots_symmetry_notmean(pvecs,means,xl=0,xu=1,yl=0,yu=1):
    
    n=5000
    
    ind = 0 
    
    clist = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown',
            'tab:pink','tab:gray','tab:olive','tab:cyan','black','lime','navy','burlywood',
            'salmon','blueviolet','chocolate','yellow','fuchsia','silver']
    plt.figure(figsize=(8,8) )  
    for pvec in pvecs: 
        l=len(pvec)
        pvec = np.array(sorted(pvec))
        seats = []
        votes = []
        small = pvec[0]
        large = pvec[-1]

        gap = large - small



        mean = means[ind] #np.mean(pvec)

        lvec = pvec - mean*np.ones([1,l])



        for t in range(n):
            tvec = lvec + (t/n)*np.ones([1,l])
            votes.append(np.mean(tvec))
            seats.append(sum(sum(tvec>=.5))/l)


        bn = np.array([mean+ (.5-x) for x in pvec])


        cn=[str(round(float(bn), 3)) for bn in bn]
        bvotes=[]
        bseats=[]

        for t in range(n):
            bvotes.append(t/n)
            bseats.append(sum(bn<(t/n))/l)


        bn = np.array([mean+ (.5-x) for x in pvec])

        bvotes=[]
        bseats=[]

        for t in range(n):
            bvotes.append(t/n)
            bseats.append(sum(bn<(t/n))/l)

        dn=list(bn[:])
        for x in bn:
            dn.append(1-x)

        rseats=list(reversed(seats))


        en=[str(round(float(dn), 3)) for dn in dn]
        area=0
        for t in range(n):
            area += (1/n)*abs(seats[t]-(1-rseats[t])) 



         
        plt.plot(bvotes,bseats,linewidth = 2,color=clist[ind],label=enames[ind],alpha=.5)
        

        #plt.plot([.5],[.5],'ro', markersize=10)
        plt.plot([mean],[sum(pvec>=.5)/l],'o',color=clist[ind], markersize=8,zorder=100)
        
        ind +=1
    
    
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')
    plt.xlabel("Democratic Vote Share")
    plt.ylabel("Democratic Seat Share")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)

    plt.xlim([xl,xu])
    plt.ylim([yl,yu])
    plt.legend()

    #plt.title("Seats -- Votes")
    
    

    plt.show()

 
def declination(vals):
    """ Compute the declination of an election.
    """
    Rwin = sorted(filter(lambda x: x <= 0.5, vals))
    Dwin = sorted(filter(lambda x: x > 0.5, vals))
    # Undefined if each party does not win at least one seat
    if len(Rwin) < 1 or len(Dwin) < 1:
        return False
    theta = np.arctan((1-2*np.mean(Rwin))*len(vals)/len(Rwin))
    gamma = np.arctan((2*np.mean(Dwin)-1)*len(vals)/len(Dwin))
    # Convert to range [-1,1]
    # A little extra precision just in case.
    return 2.0*(gamma-theta)/3.1415926535 


def lopsided_wins(vals):
    Rwin = sorted(filter(lambda x: x <= 0.5, vals))
    Dwin = sorted(filter(lambda x: x > 0.5, vals))
    
    Rmargin = abs(np.mean(Rwin)-.5)*2
    Dmargin = abs(np.mean(Dwin)-.5)*2
    
    return Rmargin - Dmargin
    

In [79]:
plan_partitions = proposed_plans
plan_part_labels = plan_names
enames = election_names

In [80]:
n_base_plans = 3

wins = [[] for i in range(n_base_plans)]
votes = [[] for i in range(n_base_plans)]
majs = [[] for i in range(n_base_plans)]
mms = [[] for i in range(n_base_plans)]
egs = [[] for i in range(n_base_plans)]
pbs =[[] for i in range(n_base_plans)]
decs = [[] for i in range(n_base_plans)]
lws = [[] for i in range(n_base_plans)]
comps = [[] for i in range(n_base_plans)]


for election in range(num_elections):
    summary = [[] for election in range(n_base_plans)]

    print(enames[election])
    #print('Votes: ', plan_partitions[0][enames[election]].percent("Democratic"))
    
    for i in range(n_base_plans):
        votes[i].append(plan_partitions[i][enames[election]].percent("Democratic"))
        summary[i].append(plan_partitions[i][enames[election]].percent("Democratic"))
        print('Votes: ', plan_partitions[i][enames[election]].percent("Democratic"))
    print('Seats')

    for i in range(n_base_plans):
        wins[i].append(plan_partitions[i][enames[election]].wins("Democratic"))
        print(plan_part_labels[i],wins[i][-1])
        summary[i].append(plan_partitions[i][enames[election]].wins("Democratic"))
    
    print("Majority?")
    for i in range(n_base_plans):
    
        if plan_partitions[i][enames[election]].percent("Democratic") > .5:
            if plan_partitions[i][enames[election]].wins("Democratic")>len(plan_partitions[i])/2:
                majs[i].append(1)
            else:
                majs[i].append(0)
        else:
            if plan_partitions[i][enames[election]].wins("Democratic")>len(plan_partitions[i])/2:
                majs[i].append(0)
            else:
                majs[i].append(1)
        print(plan_part_labels[i],majs[i][-1])
        summary[i].append(majs[i][-1])
    
    print("Mean-Median")
    for i in range(n_base_plans):
        mms[i].append(np.median(plan_partitions[i][enames[election]].percents("Democratic"))-plan_partitions[i][enames[election]].percent("Democratic"))
        print(plan_part_labels[i],mms[i][-1])
        summary[i].append(mms[i][-1])
        
    print("Partisan Bias")
    for i in range(n_base_plans):
        pbs[i].append(partisan_bias(plan_partitions[i][enames[election]]))
        print(plan_part_labels[i],pbs[i][-1])
        summary[i].append(pbs[i][-1])
    
    print("Efficiency Gap")
    for i in range(n_base_plans):
        egs[i].append(efficiency_gap(plan_partitions[i][enames[election]]))
        print(plan_part_labels[i],egs[i][-1])
        summary[i].append(egs[i][-1])
    
    
    print("Declination")
    for i in range(n_base_plans):
        decs[i].append(declination(plan_partitions[i][enames[election]].percents("Democratic")))
        print(plan_part_labels[i],decs[i][-1])
        summary[i].append(decs[i][-1])
        
    print("Lopsided Wins")
    for i in range(n_base_plans):
        lws[i].append(lopsided_wins(plan_partitions[i][enames[election]].percents("Democratic")))
        print(plan_part_labels[i],lws[i][-1])
        summary[i].append(lws[i][-1])
        
        
    print("Competitive 47-53 Districts")
    for i in range(n_base_plans):
        comps[i].append(np.sum([.47 < x <.53 for x in plan_partitions[i][enames[election]].percents("Democratic") ]))
        print(plan_part_labels[i],comps[i][-1])
        summary[i].append(comps[i][-1])
    
    df_summary = pd.DataFrame(summary, columns=['Votes', 'Seats', 'Majority', 'Mean-Median', 'Partisan Bias', 'Efficiency Gap', 'Declination', 'Lopsided Wins', 'Competitive Districts'], index=plan_part_labels)
    display(df_summary)
    df_summary.to_csv("./NC_Stats/NC_Partisan_Summary_"+enames[election]+"_eveomett.csv")
        

PRE
Votes:  0.49315799346214184
Votes:  0.49315799346214184
Votes:  0.49315799346214184
Seats
CON 3
SLDU 19
SLDL 50
Majority?
CON 1
SLDU 1
SLDL 1
Mean-Median
CON -0.056713826765421804
SLDU -0.041488458678908824
SLDL -0.04312808442824534
Partisan Bias
CON -0.2857142857142857
SLDU -0.09999999999999998
SLDL -0.08333333333333331
Efficiency Gap
CON -0.2712901568178382
SLDU -0.10920286301822116
SLDL -0.07793758555608446
Declination
CON 0.6029026769898423
SLDU 0.25309211801760245
SLDL 0.1754946523299813
Lopsided Wins
CON -0.30590508763896274
SLDU -0.12608369371337758
SLDL -0.08737405944026921
Competitive 47-53 Districts
CON 0
SLDU 4
SLDL 8


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.493158,3,1,-0.056714,-0.285714,-0.271290,0.602903,-0.305905,0
SLDU,0.493158,19,1,-0.041488,-0.100000,-0.109203,0.253092,-0.126084,4
SLDL,0.493158,50,1,-0.043128,-0.083333,-0.077938,0.175495,-0.087374,8


USS
Votes:  0.49086698030374193
Votes:  0.49086698030374193
Votes:  0.49086698030374193
Seats
CON 3
SLDU 18
SLDL 48
Majority?
CON 1
SLDU 1
SLDL 1
Mean-Median
CON -0.05114144776346036
SLDU -0.03800290856721944
SLDL -0.03887200018465192
Partisan Bias
CON -0.2857142857142857
SLDU -0.14
SLDL -0.09166666666666667
Efficiency Gap
CON -0.26517902659179154
SLDU -0.12759057621883263
SLDL -0.08845266497604938
Declination
CON 0.5814158118979424
SLDU 0.2832834485557014
SLDL 0.2002873415619581
Lopsided Wins
CON -0.269113697036799
SLDU -0.13334416727011256
SLDL -0.09347195888162729
Competitive 47-53 Districts
CON 1
SLDU 6
SLDL 5


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.490867,3,1,-0.051141,-0.285714,-0.265179,0.581416,-0.269114,1
SLDU,0.490867,18,1,-0.038003,-0.140000,-0.127591,0.283283,-0.133344,6
SLDL,0.490867,48,1,-0.038872,-0.091667,-0.088453,0.200287,-0.093472,5


GOV
Votes:  0.5228894039264802
Votes:  0.5228894039264802
Votes:  0.5228894039264802
Seats
CON 3
SLDU 23
SLDL 55
Majority?
CON 0
SLDU 0
SLDL 0
Mean-Median
CON -0.051627071162393734
SLDU -0.048017728119242487
SLDL -0.03738909288817993
Partisan Bias
CON -0.2857142857142857
SLDU -0.09999999999999998
SLDL -0.09999999999999998
Efficiency Gap
CON -0.33037981921627185
SLDU -0.09100637382076016
SLDL -0.0943261835745228
Declination
CON 0.6742709511666336
SLDU 0.1845659052822115
SLDL 0.19103481802706687
Lopsided Wins
CON -0.40841585079176457
SLDU -0.14523087320535355
SLDL -0.15219506478968847
Competitive 47-53 Districts
CON 4
SLDU 6
SLDL 19


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.522889,3,0,-0.051627,-0.285714,-0.330380,0.674271,-0.408416,4
SLDU,0.522889,23,0,-0.048018,-0.100000,-0.091006,0.184566,-0.145231,6
SLDL,0.522889,55,0,-0.037389,-0.100000,-0.094326,0.191035,-0.152195,19
